**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/lakshminarasimhan-2024"], check=True)
    os.chdir("/content/encore/entries/lakshminarasimhan-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Lakshminarasimhan et al. (2024) on dandiset 000009

**Paper.** Lakshminarasimhan KJ, Xie M, Cohen JD, Sauerbrei BA, Hantman AW, Litwin-Kumar A,
Escola S (2024). Specific connectivity optimizes learning in thalamocortical loops. *Cell Reports*
43, 114059. [doi:10.1016/j.celrep.2024.114059](https://doi.org/10.1016/j.celrep.2024.114059). The
claims file (`claims/lakshminarasimhan-2024.yaml`) was written from the NIH author manuscript in
PubMed Central (PMC11104520) with the supplement (mmc2).

**Dandiset.** [DANDI:000009](https://dandiarchive.org/dandiset/000009/0.220126.1903), version
0.220126.1903, "Maintenance of persistent activity in a frontal thalamocortical loop" (Guo et al.
2017). The paper cites the dandiset without a version (key resources table). This is the only
published version, and it is pinned for that reason. The draft holds the same 173 files with the
same blobs (checked below); it differs only in metadata.

**What the paper used from it.** The working memory data of Figure 6 and Figure S6: "5 behavioral
sessions from one mouse", 151 ALM and 72 thalamic (VM/VAL) units, correct trials, delay period. The
motor control data of the same figures (Sauerbrei et al. 2020) are not in this dandiset, nor in
any archive the paper names; a search of the DANDI API for "Sauerbrei" finds only dandiset 000221,
a different study. The simulations of Figures 1 to 5 and S1 to S5 are generated by the paper's
models and are not deposited.

**Method.** Each claim's criterion refers to the numbered method notes W1 to W10 at the head of the
claims file; the helper `tcloop.py` says for each parameter where it comes from. Random splits and
subsamples use each claim's own seed (`tcloop.seed`), and each result that depends on them is also
computed under the seed plus 1000, 2000, 3000 and 4000.

| Claim | Data | Status |
|---|---|---|
| wm-sessions-link | dandiset | tested |
| fig6-wm-choice-r2 | dandiset | tested |
| fig6c-wm-leading-pcs-drive-choice | dandiset | tested |
| figs6d-wm-thalamic-r2 | dandiset | tested |
| fig6d-wm-top5-pcs | dandiset | tested |
| figs6f-ct-alignment-not-significant | dandiset | tested |
| fig6-motor-hand-r2, fig6c-motor-low-variance-pcs, figs6d-motor-thalamic-r2, fig6d-motor-top5-pcs, fig6e-motor-single-trial | dandiset | blocked: data missing (Sauerbrei et al. 2020 not deposited) |
| fig6d-more-pcs-in-motor-task | dandiset | blocked: data missing (needs the motor data) |
| fig1c to figs5j (24 simulation claims) | elsewhere | blocked: too large (timed below), except figs5cd-thalamic-recurrence: methods underspecified |

In [ ]:
import time
import resource
import hashlib
import os
import warnings
from collections import Counter

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats

import tcloop as tc

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
print("dandiset", tc.DANDISET, "version", tc.VERSION)

## The dandiset

Every file is listed from the archive, and every extracellular file (137, about 1 MB each) is
streamed whole: its units, the region of its electrode group and its trial table. The 36
intracellular files (behavior+icephys) are from a different experiment of Guo et al. 2017 and hold
no units; only their metadata is read.

In [ ]:
assets = tc.list_assets()
draft = requests.get(f"{tc.API}/dandisets/{tc.DANDISET}/versions/draft/assets/?page_size=1000", timeout=60).json()
draft_blobs = {a["path"]: a["blob"] for a in draft["results"]}
same = sum(draft_blobs.get(a["path"]) == a["blob"] for a in assets)
print(f"{len(assets)} assets in {tc.VERSION}; {same} have the same blob in the draft ({len(draft_blobs)} draft assets)")
ecephys = sorted([a for a in assets if "ecephys" in a["path"]], key=lambda a: a["path"])
icephys = [a for a in assets if "icephys" in a["path"]]
print(len(ecephys), "extracellular files,", len(icephys), "intracellular files,",
      f"{sum(a['size'] for a in icephys) / 1e9:.1f} GB of intracellular data (not read)")
t0 = time.time()
allsess = tc.read_sessions(ecephys, workers=12)
print(f"streamed {len(allsess)} files in {time.time() - t0:.0f} s")

In [ ]:
# One file in full: every group and every column of every table.
s0 = allsess[0]
import h5py, remfile
f = h5py.File(remfile.File(tc._s3_url(ecephys[0]["asset_id"])), "r")
def walk(g, pre=""):
    for k in g:
        o = g[k]
        if isinstance(o, h5py.Dataset):
            print(f"{pre}/{k}", o.shape, o.dtype)
        else:
            walk(o, f"{pre}/{k}")
walk(f)
print("units columns:", [c.decode() for c in f["units"].attrs["colnames"]])
print("trials columns:", [c.decode() for c in f["intervals/trials"].attrs["colnames"]])
for c in f["intervals/trials"].attrs["colnames"]:
    print(f"  trials/{c.decode()}: {f['intervals/trials'][c].attrs.get('description', '')}")
f.close()

## Data check

Per subject and session: the region named by the electrode group, the number of units, and the
trials. Then the checks of the description against the values for the quantities used: trial
outcomes and photostimulation, epoch durations (delay onset to go cue), spike-time order and
range, the recording span of each unit, coordinates, and recordings deposited twice.

In [ ]:
rows = []
for s in allsess:
    t = s["trials"]
    rows.append(dict(subject=s["subject"], session=s["path"].split("_ses-")[1][:15], region=s["region"],
                     units=len(s["spikes"]), trials=len(t["id"]),
                     correct=int((t["response"] == "correct").sum()),
                     correct_nostim=int(((t["response"] == "correct") & (t["stim_present"] == 0)).sum())))
survey = pd.DataFrame(rows)
by_subject = survey.groupby(["subject", "region"]).agg(sessions=("session", "size"), units=("units", "sum")).unstack(fill_value=0)
print(by_subject.to_string())
print()
print("regions named in the files:", Counter(survey.region))
print("subjects:", survey.subject.nunique(), "sessions:", len(survey), "units:", survey.units.sum())

In [ ]:
mouse = "anm00264943"
ms = [s for s in allsess if s["subject"] == mouse]
print(survey[survey.subject == mouse].to_string(index=False))
for s in ms:
    t = s["trials"]
    delay = t["cue_start_time"] - t["pole_out_time"]
    sample = t["pole_out_time"] - t["pole_in_time"]
    ok = np.isfinite(delay)
    sp = np.concatenate(s["spikes"])
    print(f"\n{s['path'].split('/')[1]}  region {s['region']}")
    print("  type:", dict(Counter(t["type"])), " response:", dict(Counter(t["response"])))
    print("  stim_present:", dict(Counter(t["stim_present"])), " photo_stim_period:", dict(Counter(t["photo_stim_period"])),
          " photo_stim_type:", dict(Counter(t["photo_stim_type"])))
    print("  is_good among correct trials:", dict(Counter(t["is_good"][t["response"] == "correct"])))
    print(f"  delay (go cue - pole out) s: min {np.nanmin(delay):.2f} median {np.nanmedian(delay):.2f} max {np.nanmax(delay):.2f};"
          f" sample (pole out - pole in) s: min {np.nanmin(sample):.2f} median {np.nanmedian(sample):.2f} max {np.nanmax(sample):.2f};"
          f" trials without epoch times: {(~ok).sum()} (types {dict(Counter(t['type'][~ok]))})")
    print(f"  spike times {sp.min():.1f} to {sp.max():.1f} s; trials {t['start_time'].min():.1f} to {t['stop_time'].max():.1f} s;"
          f" spike trains sorted: {all(np.all(np.diff(x) >= 0) for x in s['spikes'])}")
    cov = np.array([(x[-1] - x[0]) for x in s["spikes"]]) / (np.nanmax(t["pole_out_time"]) - np.nanmin(t["pole_out_time"]))
    print(f"  unit recording span (first to last spike) as a fraction of the trial period: median {np.median(cov):.2f},"
          f" range {cov.min():.2f} to {cov.max():.2f}")
    print("  cell_type:", dict(Counter(s["units"]["cell_type"])), " unit_z values:", np.unique(np.round(s["units"]["unit_z"], 3)))
    print("  electrode group location:", s["location"])

In [ ]:
# Coordinates: the electrode-group location and the electrode positions are the same in every file.
locs = Counter(l.split("coordinate_ref")[1] for s in allsess for l in s["location"])
print("coordinate part of the electrode group location, over all files:", dict(locs))

# Recordings deposited twice: identical spike trains, and identical trial tables, across all 137 files.
h_unit, h_trial = {}, {}
for s in allsess:
    for u, x in enumerate(s["spikes"]):
        h_unit.setdefault(hashlib.sha1(np.round(x, 6).tobytes()).hexdigest(), []).append((s["path"], u))
    key = hashlib.sha1(np.round(np.nan_to_num(s["trials"]["pole_out_time"]), 4).tobytes()).hexdigest()
    h_trial.setdefault(key, []).append(s["path"])
dup_units = [v for v in h_unit.values() if len(v) > 1]
dup_trials = [v for v in h_trial.values() if len(v) > 1]
print(f"{sum(len(s['spikes']) for s in allsess)} spike trains; identical pairs: {len(dup_units)}")
for v in dup_units[:10]:
    print("  ", v)
print(f"identical trial tables (delay onsets): {len(dup_trials)}")
for v in dup_trials[:10]:
    print("  ", v)
# Within the paper's mouse, near-identical trains (same spike count and first spike) across sessions:
sig = Counter((len(x), round(float(x[0]), 3)) for s in ms for x in s["spikes"])
print("units of", mouse, "sharing spike count and first spike time:", sum(v > 1 for v in sig.values()))

### Clock

The trial table and the spikes must share a clock. Choice (lick direction of correct trials without
photostimulation) is decoded from the spike counts of each session's units in 200 ms windows around
delay onset, with a cross-validated nearest-class-mean decoder, and compared with the same decoder
on shuffled choices. ALM and thalamic activity in this task should carry little choice
information before the sample epoch (before -1.3 s), more during the sample and delay epochs, and
most after the go cue (1.3 s). A clock offset of more than a few hundred ms would move or remove
this rise.

In [ ]:
edges = np.round(np.arange(-2.6, 2.61, 0.2), 2)
rng_clock = np.random.default_rng(tc.seed("clock-check"))
clock = []
for s in ms:
    idx, ch, on = tc.select_trials(s, "T1")
    acc = tc.choice_decoding(s, on, ch, edges, rng_clock)
    null = np.mean([tc.choice_decoding(s, on, rng_clock.permutation(ch), edges, rng_clock) for _ in range(5)], 0)
    clock.append(acc)
    pre = acc[edges[:-1] < -1.4].mean(); dl = acc[(edges[:-1] >= 0) & (edges[:-1] < 1.3)].mean()
    rsp = acc[edges[:-1] >= 1.4].mean()
    print(f"{s['region']:8s} {s['path'].split('_ses-')[1][:15]}  n={len(ch)}  accuracy before sample {pre:.2f}, delay {dl:.2f}, "
          f"after go cue {rsp:.2f}; shuffled choices {null.mean():.2f} (max over windows {null.max():.2f})")
clock = np.array(clock)
fig, ax = plt.subplots(figsize=(7, 3))
for a, s in zip(clock, ms):
    ax.plot(edges[:-1] + 0.1, a, color="C0" if s["region"] == "ALM" else "C1", alpha=0.7)
for x in (-1.3, 0, 1.3):
    ax.axvline(x, color="k", lw=0.5)
ax.set_xlabel("time from delay onset (s)"); ax.set_ylabel("choice decoding accuracy"); ax.set_ylim(0.3, 1.0)
ax.set_title("ALM (blue) and thalamus (orange) sessions of " + mouse)
plt.show()
print(f"plotted accuracy ranges from {clock.min():.2f} to {clock.max():.2f}; none outside the axes")

## wm-sessions-link

> The dandiset holds the working memory data the paper reanalyzed, five sessions from one mouse with
> 151 ALM units and 72 thalamic units in total.

Criterion: among the mice with ALM and thalamic units, find one with five sessions whose unit totals
are within tolerance (151 +/- 15 ALM, 72 +/- 7 thalamic) under a reading of W1. All sorted units
are counted (the files carry no isolation-quality label; `cell_type` is a spike-width class).

In [ ]:
both = by_subject[(by_subject[("units", "ALM")] > 0) & (by_subject[("units", "Thalamus")] > 0)]
print("mice with both ALM and thalamic units:")
print(both.to_string())
fit = []
for subj, r in both.iterrows():
    alm_n, th_n = r[("units", "ALM")], r[("units", "Thalamus")]
    ok = abs(alm_n - 151) <= 15 and abs(th_n - 72) <= 7
    fit.append(ok)
    print(f"{subj}: ALM {alm_n} units in {r[('sessions', 'ALM')]} sessions, thalamus {th_n} units in "
          f"{r[('sessions', 'Thalamus')]} sessions; within tolerance: {ok}")
print("mice that fit:", int(np.sum(fit)))
# Other regions named in the files, for completeness:
print("mice with SNr units:", sorted(survey[survey.region == "SNr"].subject.unique()))
alm_sessions = [s for s in ms if s["region"] == "ALM"]
thal_sessions = [s for s in ms if s["region"] == "Thalamus"]
both_sessions = [s for s in ms if {"ALM", "Thalamus"} <= set(s["region"].split(";"))]
print(f"\n{mouse}: {len(alm_sessions)} ALM sessions with {sum(len(s['spikes']) for s in alm_sessions)} units "
      f"({[len(s['spikes']) for s in alm_sessions]}); {len(thal_sessions)} thalamic sessions with "
      f"{sum(len(s['spikes']) for s in thal_sessions)} units ({[len(s['spikes']) for s in thal_sessions]}); "
      f"sessions holding both: {len(both_sessions)}")
n_sess = len(ms)
print(f"sessions of {mouse} (expected 5, tolerance 0): {n_sess}; within tolerance: {n_sess == 5}")
print(f"ALM units {sum(len(s['spikes']) for s in alm_sessions)} (151 +/- 15), thalamic units "
      f"{sum(len(s['spikes']) for s in thal_sessions)} (72 +/- 7); ALM sessions alone: {len(alm_sessions)}")

**Result: partially reproduced.** One mouse (anm00264943) has both ALM and thalamic units, and
its totals are the paper's exactly: 151 ALM units in five sessions and 72 thalamic units. The
thalamic units come from six further sessions, so the mouse has eleven sessions, not the five the
expected value requires (tolerance 0), and none holds both regions. The paper's "5 behavioral
sessions" matches the five ALM sessions (Figure 6C, n = 5). The only other mouse with ALM units
beside units of another region (anm00261335) has SNr, not thalamic, units. All sorted units are
counted. The link holds for the mouse and the units; the dependent claims are tested on these
eleven sessions, and the six thalamic sessions are paired with the ALM sessions as described below.

## Preparing the paper's mouse

Correct trials, under reading T1 (no photostimulation) and T2 (all correct trials), aligned to delay
onset, binned at 10 ms and smoothed at each sigma of the grid (helper docstring).

In [ ]:
t0 = time.time()
PREP = {r: [tc.Prepared(s, r) for s in ms] for r in ("T1", "T2")}
ALM = {r: [p for p in PREP[r] if p.region == "ALM"] for r in PREP}
THAL = {r: [p for p in PREP[r] if p.region == "Thalamus"] for r in PREP}
for r in PREP:
    print(r, "trials per session (left, right):",
          [(int((p.choice == 0).sum()), int((p.choice == 1).sum())) for p in PREP[r]])
    for p in PREP[r]:
        x = p.rates[50]
        assert np.isfinite(x).all()
    print(f"  rates at sigma 50 ms: {min(p.rates[50].min() for p in PREP[r]):.1f} to "
          f"{max(p.rates[50].max() for p in PREP[r]):.1f} spikes/s;"
          f" fraction of unit-trial pairs inside the unit's span: "
          f"{np.mean(np.concatenate([p.span.ravel() for p in PREP[r]])):.2f}")
print(f"prepared in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## fig6-wm-choice-r2

> In the working memory task, ALM population activity explains the animal's choice well (R2 = 0.97).

Criterion: test-set R2 of c(t) predicted from the test-set h(t) with the training-set weights,
under R1 (per session, mean over the five sessions) and R2 (pooled 151 units subsampled to 53, mean
over 20 subsamples), trial readings T1 and T2, mean over 10 splits. Reproduced when within 0.97 +/-
0.097 under one reading. The unit-span reading U2 and the training-set R2 are shown as variants;
neither decides the outcome.

In [ ]:
CL = "fig6-wm-choice-r2"
choice_res = {}
for off in (0, 1000, 2000, 3000, 4000):
    for r in ("T1", "T2"):
        for span in ((False, True) if off == 0 else (False,)):
            rng = np.random.default_rng(tc.seed(CL, off))
            r1 = tc.readout_per_session(ALM[r], rng, span=span)
            r2p = tc.readout_pooled(ALM[r], rng, span=span)
            choice_res[(off, r, span)] = (r1, r2p)
rows = []
for (off, r, span), (r1, r2p) in choice_res.items():
    rows.append(dict(seed_offset=off, trials=r, units="U2" if span else "U1",
                     R1_test_mean=np.mean([d["test"].mean() for d in r1]),
                     R1_test_median=np.median(np.concatenate([d["test"] for d in r1])),
                     R1_one_split=np.mean([d["test"][0] for d in r1]),
                     R1_train=np.mean([d["train"].mean() for d in r1]),
                     R2_test_mean=r2p["test"].mean(), R2_test_median=np.median(r2p["test"]),
                     R2_one_split=r2p["test"][:tc.N_SUB].mean(), R2_train=r2p["train"].mean()))
choice_tab = pd.DataFrame(rows)
print(choice_tab.round(3).to_string(index=False))

In [ ]:
r1, r2p = choice_res[(0, "T1", False)]
print("Claim seed, T1, U1. Per session (R1):")
for d in r1:
    print(f"  {d['path'].split('_ses-')[1][:15]}  units {d['n'][0]}  test R2 mean {d['test'].mean():.2f} median {np.median(d['test']):.2f}"
          f" (range {d['test'].min():.2f} to {d['test'].max():.2f}); validation {d['val'].mean():.2f}; training {d['train'].mean():.3f};"
          f" sigma chosen {dict(Counter(d['sigma']))}")
print(f"Pooled (R2): sigma chosen {dict(Counter(r2p['sigma']))}; fraction of fits with test R2 > 0: {(r2p['test'] > 0).mean():.2f}")
main = choice_tab[(choice_tab.seed_offset == 0) & (choice_tab.units == "U1")]
best = max(main[["R1_test_mean", "R2_test_mean"]].max())
print(f"\nlargest test R2 over the four readings (R1/R2 x T1/T2), claim seed: {best:.3f}; tolerance band 0.873 to 1")
u1 = choice_tab[choice_tab.units == "U1"]
print(f"over five seeds, readings R1/R2 x T1/T2: test R2 means range {u1[['R1_test_mean', 'R2_test_mean']].min().min():.2f} to "
      f"{u1[['R1_test_mean', 'R2_test_mean']].max().max():.2f}; medians {u1[['R1_test_median', 'R2_test_median']].min().min():.2f} to "
      f"{u1[['R1_test_median', 'R2_test_median']].max().max():.2f}")

In [ ]:
# Diagnostics, not readings of the claim: test R2 against the size of the test set, and how
# reproducible the readout's output on the test set is. Pooled readout (R2), T1, U1, with a second
# generator of the claim (seed + 1). Validation stays at 10%; the training set shrinks.
rng = np.random.default_rng(tc.seed(CL, 1))
for tf in (0.1, 0.3, 0.5):
    d = tc.readout_pooled(ALM["T1"], rng, test_frac=tf)
    r = np.nanmean(d["rel"])
    print(f"test fraction {tf:.1f}: test R2 mean {d['test'].mean():.3f}, median {np.median(d['test']):.3f}, "
          f"fraction > 0 {(d['test'] > 0).mean():.2f}; training R2 {d['train'].mean():.3f}; correlation of the readout's "
          f"outputs on the two halves of the test set {r:.3f} (reliability of the whole test set {2 * r / (1 + r):.3f})")

**Result: not reproduced.** Under every reading the test-set R2 of the choice readout is far below
0.97. With every unit averaged over all selected trials (U1), under the claim seed, the largest of
the four means (R1 or R2, T1 or T2) is -0.247, and the medians run from -0.86 to 0.34; over five
seeds no U1 mean exceeds 0.105. The readout fits the training averages almost perfectly (training R2
0.95 to 1.00 under U1, 0.93 to 1.00 under U2) and does not generalize to the test averages of 4 to 9
trials per choice. The U2 variant gives means of at most 0.506 and medians of at most 0.660. In the diagnostics (seed + 1, R2, T1),
the readout's outputs on the two halves of a 10% test set correlate at 0.145 (reliability of the
whole test set 0.25), and larger test sets do not bring the test R2 near 0.97: the means are -0.089,
0.371 and -0.333 and the medians 0.268, 0.515 and 0.313 for 10%, 30% and 50% test sets. The
training-set R2 is close to the paper's 0.97; whether the paper's value was computed on the trials
used to fit is asked of the paper's authors, not a reading of the claim.

## fig6c-wm-leading-pcs-drive-choice

> In the working memory task, the leading PCs of ALM activity have a large influence on the readout
> of choice.

Criterion: alignment |w . u_i| / ||w|| of the readout with PCs 1 to 16, per session with the mean
over the five sessions (R1), and on the pooled subsampled population (R2). Readings (a) the PC with
the largest mean alignment is PC 1 or 2; (b) the mean alignment over PCs 1 to 3 exceeds the mean
over PCs 4 to 16. Reproduced when both hold under one trial reading, partially when one holds.

In [ ]:
CL = "fig6c-wm-leading-pcs-drive-choice"
pc_rows, profiles, pc_res = [], {}, {}
for off in (0, 1000, 2000, 3000, 4000):
    for r in ("T1", "T2"):
        for span in ((False, True) if off == 0 else (False,)):
            rng = np.random.default_rng(tc.seed(CL, off))
            r1 = tc.readout_per_session(ALM[r], rng, span=span)
            r2p = tc.readout_pooled(ALM[r], rng, span=span)
            pc_res[(off, r, span)] = (r1, r2p)
            for name, prof, sem in (("R1", np.nanmean([np.nanmean(d["align"], 0) for d in r1], 0),
                                     stats.sem([np.nanmean(d["align"], 0) for d in r1], 0, nan_policy="omit")),
                                    ("R2", np.nanmean(r2p["align"], 0), np.nanstd(r2p["align"], 0) / np.sqrt(len(r2p["align"])))):
                profiles[(off, r, span, name)] = (prof, sem)
                a = int(np.argmax(prof)) + 1
                b = prof[:3].mean() > prof[3:].mean()
                pc_rows.append(dict(seed_offset=off, trials=r, units="U2" if span else "U1", readout=name,
                                    argmax_PC=a, mean_PC1_3=prof[:3].mean(), mean_PC4_16=prof[3:].mean(),
                                    a_holds=a <= 2, b_holds=bool(b)))
pc_tab = pd.DataFrame(pc_rows)
print(pc_tab.round(4).to_string(index=False))
r1t1 = pc_tab[(pc_tab.readout == "R1") & (pc_tab.trials == "T1") & (pc_tab.units == "U1")]
print("R1, T1, U1, reading (b): mean over PCs 1-3 minus mean over PCs 4-16, by seed offset:",
      {int(k): float(f"{v:.2g}") for k, v in zip(r1t1.seed_offset, r1t1.mean_PC1_3 - r1t1.mean_PC4_16)})

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
lo, hi = np.inf, -np.inf
for ax, name in zip(axs, ("R1", "R2")):
    for r, c in (("T1", "C0"), ("T2", "C2")):
        prof, sem = profiles[(0, r, False, name)]
        ax.errorbar(np.arange(1, 17), prof, sem, color=c, label=r, marker="o", ms=3)
        lo, hi = min(lo, (prof - sem).min()), max(hi, (prof + sem).max())
    ax.set_title(f"readout {name}, claim seed, U1"); ax.set_xlabel("PC"); ax.legend()
axs[0].set_ylabel("|w . u_i| / ||w||")
plt.show()
print(f"plotted range {lo:.3f} to {hi:.3f}")
for name in ("R1", "R2"):
    for r in ("T1", "T2"):
        prof, sem = profiles[(0, r, False, name)]
        print(name, r, "alignment PCs 1-16:", np.round(prof, 3))
        print(name, r, "SEM:", np.round(sem, 3))
m = pc_tab[(pc_tab.seed_offset == 0) & (pc_tab.units == "U1")]
print("\nclaim seed, U1: readings where (a) holds:", m[m.a_holds][["trials", "readout"]].values.tolist(),
      "; where (b) holds:", m[m.b_holds][["trials", "readout"]].values.tolist(),
      "; where both hold:", m[m.a_holds & m.b_holds][["trials", "readout"]].values.tolist())
s5 = pc_tab[pc_tab.units == "U1"].groupby("seed_offset").apply(lambda g: bool((g.a_holds & g.b_holds).any()))
print("both hold under some reading, per seed offset:", s5.to_dict())
s5a = pc_tab[pc_tab.units == "U1"].groupby("seed_offset").apply(lambda g: bool((g.a_holds | g.b_holds).any()))
print("at least one holds under some reading, per seed offset:", s5a.to_dict())
# Reference: alignment |w . u| / ||w|| of a random direction w with a fixed unit vector u, in d dimensions.
rng_ref = np.random.default_rng(tc.seed(CL, 1))
for d in sorted({p.n_units for p in ALM["T1"]} | {tc.N_KEEP}):
    x = rng_ref.standard_normal((20000, d))
    print(f"random-direction reference, d = {d}: mean alignment {np.mean(np.abs(x[:, 0]) / np.linalg.norm(x, axis=1)):.3f}")

In [ ]:
# The readout's alignment with every PC, against chance. The squared alignments over all d PCs sum to 1,
# so a random direction has a mean squared alignment of 1/d with each PC and puts a share 16/d of its
# squared norm in PCs 1 to 16.
for r in ("T1", "T2"):
    r1, r2p = pc_res[(0, r, False)]
    sq = r2p["align_all"] ** 2
    d = sq.shape[1]
    top = sq[:, :16].sum(1)
    print(f"R2 {r}: d = {d}; mean squared alignment with PCs 1-3 {sq[:, :3].mean():.4f}, 4-16 {sq[:, 3:16].mean():.4f}, "
          f"17-{d} {sq[:, 16:].mean():.4f}, chance 1/d {1 / d:.4f}; share of the readout's squared norm in PCs 1-16 "
          f"{top.mean():.3f} (range {top.min():.3f} to {top.max():.3f}), chance 16/d {16 / d:.3f}")
    for dd in r1:
        n = int(dd["n"][0])
        print(f"  R1 {r} {dd['path'].split('_ses-')[1][:15]}: {n} units; share in PCs 1-16 {dd['top16'].mean():.3f}, "
              f"chance {min(16, n) / n:.3f}")
sq = pc_res[(0, "T1", False)][1]["align_all"] ** 2
print("R2 T1, mean squared alignment by blocks of PCs (1-8, 9-16, ..., 49-53):",
      np.round([sq[:, a:a + 8].mean() for a in range(0, sq.shape[1], 8)], 4))

**Result: partially reproduced.** The criterion names both readouts, per session (R1, the reading of
Figure 6C, n = 5) and pooled and subsampled (R2), and lets only the trial reading vary. Under R2
both readings of "leading" hold for T1 and T2 under the claim seed: the largest alignment is with
PC 2 (T1) or PC 1 (T2), and the mean over PCs 1 to 3 exceeds the mean over PCs 4 to 16 (0.081
against 0.063 for T1). Both hold under some reading for all five seeds. Under R1 reading (a) fails
for both trial readings, with the largest alignment at PC 16, and (b) holds only for T1, by a
margin of 0.0003 (0.1069 against 0.1065); it fails under seed + 2000 (by 0.0015) and holds under the
other three seeds. Were (b) to fail under R1 as well, R1 would meet neither reading while R2 meets
both, so the claim would still hold only in part. Measured against chance, the leading PCs carry less of the readout than a random
direction would. Under R2 the mean squared alignment with PCs 1 to 3 is 0.0096 (T1) and 0.0076 (T2),
against 1/53 = 0.0189, and PCs 1 to 16 hold 0.116 and 0.113 of the readout's squared norm, against
16/53 = 0.302. Per session that share is below its chance level in all five sessions under both
trial readings (0.045 to 0.593, against 0.348 to 0.889). The readout lies mostly in PCs 17 to 53,
and it does not predict test data (fig6-wm-choice-r2).

## Thalamic regressions (figs6d-wm-thalamic-r2, fig6d-wm-top5-pcs, figs6f-ct-alignment-not-significant)

Each thalamic unit's trial-averaged r(t) (its own session's correct trials, split 80/10/10) is
regressed on the trial-averaged ALM population of the five ALM sessions (W9): P1, the 151 pooled
units subsampled to 53, 20 subsamples per split; P2, all 151. P3 (each thalamic unit with the ALM
units of its own session) does not apply: no session holds both. Each claim runs with its own seed;
under the claim seed every reading (T1/T2, P1/P2, U1/U2) is run, and under the four further seeds
the readings T1/T2 x P1/P2 with U1.

In [ ]:
def run_thal(claim):
    out = {}
    for off in (0, 1000, 2000, 3000, 4000):
        for r in ("T1", "T2"):
            for pool in ("P1", "P2"):
                for span in ((False, True) if off == 0 else (False,)):
                    rng = np.random.default_rng(tc.seed(claim, off))
                    out[(off, r, pool, span)] = tc.thalamic_analysis(ALM[r], THAL[r], rng, pool=pool, span=span)
    return out

t0 = time.time()
TH = {c: run_thal(c) for c in ("figs6d-wm-thalamic-r2", "fig6d-wm-top5-pcs", "figs6f-ct-alignment-not-significant")}
print(f"thalamic regressions in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

### figs6d-wm-thalamic-r2

> ALM activity explains 0.70 +/- 0.01 of the variance of VM/VAL thalamic neurons in the working
> memory task.

Criterion: test-set R2 from all cortical units of the reading of W9 (53 units in P1), mean over the
72 thalamic units; reproduced when within 0.70 +/- 0.01 under one combination of T and P.

In [ ]:
res = TH["figs6d-wm-thalamic-r2"]
rows = []
for (off, r, pool, span), d in res.items():
    rows.append(dict(seed_offset=off, trials=r, pool=pool, units="U2" if span else "U1",
                     mean_test_R2=np.nanmean(d["full"]), median_test_R2=np.nanmedian(d["full"]),
                     sem=stats.sem(d["full"], nan_policy="omit"), n_units=int(np.isfinite(d["full"]).sum()),
                     frac_positive=np.nanmean(d["full"] > 0), min_fits=int(d["n_fits"].min()),
                     mean_train_R2=np.nanmean(d["full_train"])))
s6d = pd.DataFrame(rows)
print(s6d.round(3).to_string(index=False))
m = s6d[(s6d.seed_offset == 0) & (s6d.units == "U1")]
print("\nclaim seed, U1: combinations within 0.69 to 0.71:", m[(m.mean_test_R2 - 0.70).abs() <= 0.01][["trials", "pool"]].values.tolist())
print(f"test R2 means over all seeds and U1 readings: {s6d[s6d.units == 'U1'].mean_test_R2.min():.2f} to {s6d[s6d.units == 'U1'].mean_test_R2.max():.2f}")
d = res[(0, "T1", "P1", False)]
fig, ax = plt.subplots(figsize=(5, 3))
x = np.sort(d["full"]); ax.plot(np.clip(x, -5, 1), np.arange(1, len(x) + 1) / len(x))
ax.set_xlabel("test R2 from all cortical units (clipped at -5)"); ax.set_ylabel("cumulative fraction")
plt.show()
print(f"T1 P1 U1: test R2 range {x.min():.2f} to {x.max():.2f}; {np.sum(x < -5)} units below -5 drawn at -5")

In [ ]:
# Diagnostics, not readings of the claim, with a second generator of the claim (seed + 1).
# (1) Noise ceilings of the thalamic test-set averages, at each sigma: the R2 that each unit's own
# training-set average reaches on its test-set average, and the correlation r between the averages of
# the two halves of the test set (Spearman-Brown reliability of the whole test set, 2r / (1 + r)).
# (2) The main reading (T1, P1, U1) with 30% and 50% test sets (training 60% and 40%).
rng = np.random.default_rng(tc.seed("figs6d-wm-thalamic-r2", 1))
for tf in (0.1, 0.5):
    ceil = tc.thalamic_ceiling(THAL["T1"], rng, test_frac=tf)
    for s in tc.SIGMAS_MS:
        r = np.nanmean(ceil[s]["half"])
        print(f"test fraction {tf}, sigma {s:3d} ms: R2 of the test average from the training average, mean over units "
              f"{np.nanmean(ceil[s]['train']):.3f} (median {np.nanmedian(ceil[s]['train']):.3f}, max {np.nanmax(ceil[s]['train']):.3f}); "
              f"half correlation mean {r:.3f}, reliability {2 * r / (1 + r):.3f}")
for tf in (0.3, 0.5):
    d = tc.thalamic_analysis(ALM["T1"], THAL["T1"], rng, pool="P1", test_frac=tf)
    print(f"T1 P1 U1, test fraction {tf}: mean test R2 from all cortical units {np.nanmean(d['full']):.3f} "
          f"(median {np.nanmedian(d['full']):.3f}, fraction positive {np.nanmean(d['full'] > 0):.2f}); "
          f"training R2 {np.nanmean(d['full_train']):.3f}")

**Result: not reproduced.** No combination of trial and pooling readings gives a mean test-set R2
near 0.70: under the claim seed the means run from -5.16 to -1.91, and no unit has a positive test
R2 under P1. The training-set R2 of the same regressions, a diagnostic only, is 0.73 to 0.77 under
P1 over the five seeds and near 1 under P2. The diagnostics (seed + 1, T1) show that a mean test-set
R2 near 0.70 is out of reach with 10% test sets: the averages of the two halves of a unit's test set
correlate at 0.04 to 0.17 on average over the sigma grid (reliability of the whole test set 0.07 to
0.28), and a unit's own training-set average predicts its test-set average with a mean R2 of -0.05
to 0.06. A model that predicted each unit's mean exactly would explain on average at most about 0.28
of the variance of its test-set average. With 30% and 50% test sets the mean test R2 of the main
reading is -1.57 and -1.72.

### fig6d-wm-top5-pcs

> In the working memory task, the top 5 ALM PCs capture 0.63 +/- 0.04 of the explainable variance of
> thalamic neurons.

Criterion: mean normalized R2(5) over the 72 thalamic units under N1 (R2(5)/R2(16)), N2 (R2(5)/R2
from all cortical units) or N3 (each unit's curve rescaled to 0 to 1); reproduced when within 0.63
+/- 0.04 under one combination of T, P and N. Normalization is applied to each unit's mean curve
over splits and subsamples.

In [ ]:
res = TH["fig6d-wm-top5-pcs"]
rows = []
for (off, r, pool, span), d in res.items():
    for how in ("N1", "N2", "N3"):
        v = tc.normalized(d["r2k"], d["full"], how)[4]
        vt = tc.normalized(d["r2k_train"], d["full_train"], how)[4]
        rows.append(dict(seed_offset=off, trials=r, pool=pool, units="U2" if span else "U1", norm=how,
                         mean=np.nanmean(v), median=np.nanmedian(v), sem=stats.sem(v, nan_policy="omit"),
                         n=int(np.isfinite(v).sum()), mean_train=np.nanmean(vt)))
t5 = pd.DataFrame(rows)
print(t5.round(3).to_string(index=False))
m = t5[(t5.seed_offset == 0) & (t5.units == "U1")]
hit = m[(m["mean"] - 0.63).abs() <= 0.04]
print("\nclaim seed, U1: combinations within 0.59 to 0.67:", hit[["trials", "pool", "norm"]].values.tolist())
for s, g in t5[t5.units == "U1"].groupby("seed_offset"):
    print(f"seed offset {s}: combinations within tolerance: {g[(g['mean'] - 0.63).abs() <= 0.04][['trials', 'pool', 'norm']].values.tolist()}")
d = res[(0, "T1", "P1", False)]
print("\nT1 P1 U1, mean test R2(k), k = 1 to 16:", np.round(np.nanmean(d["r2k"], 1), 3))
print("T1 P1 U1, mean training R2(k), k = 1 to 16:", np.round(np.nanmean(d["r2k_train"], 1), 3))
print("units whose test R2(16) is positive:", int(np.sum(d["r2k"][-1] > 0)), "of", d["r2k"].shape[1])

In [ ]:
# N1 and N2 are fractions of explainable variance only where the denominator is positive.
for key in [k for k in res if k[0] == 0 and not k[3]]:
    d = res[key]
    pos1, pos2 = d["r2k"][-1] > 0, d["full"] > 0
    n1 = d["r2k"][4, pos1] / d["r2k"][-1, pos1]
    n2 = d["r2k"][4, pos2] / d["full"][pos2]
    print(f"{key[1]} {key[2]} U1: units with test R2(16) > 0: {pos1.sum()} of {len(pos1)} (N1 over them: "
          f"{n1.mean() if len(n1) else float('nan'):.3f}); with test R2 from all units > 0: {pos2.sum()} (N2 over them: "
          f"{n2.mean() if len(n2) else float('nan'):.3f})")
# N3 rescales each curve between its minimum and maximum; where R2(k) falls with k it measures that fall.
d = res[(0, "T1", "P1", False)]
print("T1 P1 U1: units whose test R2(k) is largest at k = 1:", int(np.sum(np.argmax(d["r2k"], 0) == 0)),
      "of", d["r2k"].shape[1])
# A diagnostic, not a reading of the claim: the main reading with 50% test sets (training 40%), with a
# second generator of the claim (seed + 1).
rng = np.random.default_rng(tc.seed("fig6d-wm-top5-pcs", 1))
d5 = tc.thalamic_analysis(ALM["T1"], THAL["T1"], rng, pool="P1", test_frac=0.5)
print("\nT1 P1 U1, 50% test sets: mean test R2(k), k = 1 to 16:", np.round(np.nanmean(d5["r2k"], 1), 3))
print("units whose test R2(16) is positive:", int(np.sum(d5["r2k"][-1] > 0)), "of", d5["r2k"].shape[1])
for how in ("N1", "N2", "N3"):
    print(f"  {how}: mean normalized R2(5) {np.nanmean(tc.normalized(d5['r2k'], d5['full'], how)[4]):.3f}")

**Result: not reproduced.** No combination of T, P and N gives a mean normalized R2(5) within 0.59
to 0.67, under any of the five seeds. The test-set R2(k) is near 0 at k = 1 (0.005 under T1 and P1)
and falls as k grows (the regressions on more PCs overfit). Over the five seeds with U1, N1 gives at
most 0.31 under P1 and means from -1.02 to 2.01, driven by outliers, under P2; N2 gives -0.09 to
0.11; and N3 gives 0.76 to 0.86. None of these is a fraction of explainable variance. Under the claim
seed no unit has a positive test R2(16) under P1 (6 under T1 and P2, 10 under T2 and P2), and no unit
has a positive test R2 from all units under any reading, so the denominators of N1 and N2 are not
positive. N3 rescales each curve between its minimum and its maximum; 48 of 72 units have their
largest test R2(k) at k = 1 (T1, P1), so its value at k = 5 measures the fall of the curve. With 50%
test sets (a diagnostic) the mean test R2(k) peaks at 0.043 (k = 3), and N1, N2 and N3 give 3.89,
-0.16 and 0.87. On the training set, a diagnostic only, N2 gives 0.68 and N3 0.70 under T1 and P1.

### figs6f-ct-alignment-not-significant

> For individual thalamic neurons, the difference between the alignment of corticothalamic weights
> with the leading cortical PC and with the readout direction is not significant in either dataset
> (one-sided paired t tests, p = 0.09 and p = 0.17).

Criterion: a claim of absence on the paper's test. Corticothalamic weights from the regression on
all 16 PCs mapped back to unit space; alignment |cos| with PC 1 and with the readout on the same
cortical population; one-sided paired t test over the 72 units, PC greater than readout;
reproduced when p >= 0.05. The cortical population follows the Figure S6 legend ("in all
analysis", P1); T1 is the main trial reading (W2 lists it first). The other readings and the
weights of the regression on all units are variants.

In [ ]:
res = TH["figs6f-ct-alignment-not-significant"]
rows = []
for (off, r, pool, span), d in res.items():
    for w in ("16 PCs", "all units"):
        a, b = (d["a_pc"], d["a_ro"]) if w == "16 PCs" else (d["a_pc_full"], d["a_ro_full"])
        ok = np.isfinite(a) & np.isfinite(b)
        t = stats.ttest_rel(a[ok], b[ok], alternative="greater")
        t2 = stats.ttest_rel(a[ok], b[ok])
        diff = a[ok] - b[ok]
        rows.append(dict(seed_offset=off, trials=r, pool=pool, units="U2" if span else "U1", weights=w,
                         mean_align_PC1=a[ok].mean(), mean_align_readout=b[ok].mean(),
                         mean_diff=diff.mean(), cohen_dz=diff.mean() / diff.std(ddof=1), t=t.statistic, p=t.pvalue,
                         p_two_sided=t2.pvalue, n=int(ok.sum())))
s6f = pd.DataFrame(rows)
print(s6f.round(4).to_string(index=False))
main = s6f[(s6f.seed_offset == 0) & (s6f.trials == "T1") & (s6f.pool == "P1") & (s6f.units == "U1") & (s6f.weights == "16 PCs")].iloc[0]
print(f"\nmain reading (claim seed, T1, P1, U1, 16 PCs): PC1 {main.mean_align_PC1:.3f}, readout {main.mean_align_readout:.3f}, "
      f"difference {main.mean_diff:.3f}, t = {main.t:.2f}, one-sided p = {main.p:.2g}, n = {main.n}")
sd = s6f[(s6f.trials == "T1") & (s6f.pool == "P1") & (s6f.units == "U1") & (s6f.weights == "16 PCs")]
print("main reading over five seeds, p:", sd.set_index("seed_offset").p.round(4).to_dict())
print("readings with p < 0.05 (claim seed):", s6f[(s6f.seed_offset == 0) & (s6f.p < 0.05)][["trials", "pool", "units", "weights"]].values.tolist())
print("readings with p >= 0.05 (any seed):", s6f[s6f.p >= 0.05][["seed_offset", "trials", "pool", "units", "weights"]].values.tolist())
rng_ref = np.random.default_rng(tc.seed("figs6f-ct-alignment-not-significant", 1))
for d in (16, tc.N_KEEP, 151):
    x = rng_ref.standard_normal((20000, d))
    print(f"random-direction reference, d = {d}: mean alignment {np.mean(np.abs(x[:, 0]) / np.linalg.norm(x, axis=1)):.3f}")

In [ ]:
# Chance levels (a diagnostic, not a reading of the claim). The 16-PC weights lie in the span of PCs 1
# to 16. A random direction in that span has mean alignment c16 with PC 1, and c16 times the share of
# the readout's norm inside the span with the readout (c_ro, per unit and fit). The comparison with
# each alignment divided by its chance level is run with the readout at its own best sigma (main) and
# at the thalamic unit's sigma (same sigma).
c16 = tc.chance_alignment(16)
print(f"c16 = {c16:.3f}")
rows = []
for (off, r, pool, span), d in res.items():
    if off:
        continue
    for ro in ("main", "same sigma"):
        a_ro, c_ro = (d["a_ro"], d["c_ro"]) if ro == "main" else (d["a_ro_same"], d["c_ro_same"])
        x, y = d["a_pc"] / c16, a_ro / c_ro
        ok = np.isfinite(x) & np.isfinite(y)
        raw = stats.ttest_rel(d["a_pc"][ok], a_ro[ok], alternative="greater")
        rel = stats.ttest_rel(x[ok], y[ok])
        rows.append(dict(trials=r, pool=pool, units="U2" if span else "U1", readout_sigma=ro,
                         share_of_readout_in_span=np.nanmean(d["ro16"]), PC1=np.nanmean(d["a_pc"]), PC1_chance=c16,
                         readout=np.nanmean(a_ro), readout_chance=np.nanmean(c_ro),
                         p_raw_greater=raw.pvalue, PC1_over_chance=np.nanmean(x), readout_over_chance=np.nanmean(y),
                         dz_over_chance=np.nanmean(x[ok] - y[ok]) / np.nanstd(x[ok] - y[ok], ddof=1),
                         p_over_chance_two_sided=rel.pvalue))
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: not reproduced.** On the main reading (T1, P1, U1, weights from the 16 PCs) the
corticothalamic weights are more aligned with PC 1 than with the readout (0.094 against 0.071, mean
difference 0.023, dz 0.54, t = 4.62, one-sided p = 8.3e-06, n = 72), under all five seeds, so the
paper's test finds the difference the claim says it does not. Every reading with the 16-PC weights
gives p < 0.05 under the claim seed; only T2 with P1 gives p >= 0.05 under two of the other seeds
(0.21 and 0.052). With the weights of the regression on all units, a variant, the readout alignment
is the larger one (one-sided p = 1, two-sided p < 0.001, dz -10.5), which satisfies the criterion's
p >= 0.05 by an effect in the other direction. Measured against chance within the 16-PC span, where
the 16-PC weights lie, the order reverses: the PC 1 alignment is 0.46 of its chance level (0.094
against 0.203) and the readout alignment 1.19 times its own (0.071 against 0.059; the readout has
0.29 of its norm in the span), a chance-corrected difference of dz -2.36 (two-sided p < 0.001). The
same holds with the readout taken at each thalamic unit's sigma (ratio 1.21, raw one-sided p < 0.001).
With the readout at the thalamic unit's sigma, one other reading, T2 with P1 and U1, gives a raw
one-sided p of 0.140 (cell above); this variant does not decide the outcome.
The paper's test thus compares two alignments with different chance levels; the outcome follows the
test as the criterion gives it.

## The simulation claims

The 24 claims on the paper's models (Figures 1 to 5, S1 to S5) need the models run as the paper
ran them: 10,000 learning trials of 200 time steps for each of 40 initializations, for every
network size and condition, and, for the meta-learning claims, 10,000 epochs of 200 such trials
each. One epoch of the paper's own motor control model (`thalamocortical__motor.py`, commit 2aa94dd,
reimplemented in `tcloop.rflo_motor`) is timed here at the smallest and largest thalamus.

In [ ]:
rng = np.random.default_rng(tc.seed("fig1c-tc-learning-improves-with-thalamus"))
per_trial = {}
tc.rflo_motor(1, 1, rng)  # untimed first call, so that the timings below exclude start-up costs
def timed(M, N=256):
    # Fastest of three timings of 3 learning trials, so the estimates below are lower bounds.
    best = np.inf
    for _ in range(3):
        t0 = time.time(); e = tc.rflo_motor(3, M, rng, N=N); best = min(best, (time.time() - t0) / 3)
    return best, e
for M in (1, 8, 32, 128, 256):
    per_trial[M], e = timed(M)
    print(f"M = {M}: {per_trial[M] * 1000:.1f} ms per learning trial; errors of the last 3 trials {np.round(e, 3)}")
n512, _ = timed(1, N=512)
print(f"N = 512, M = 1: {n512 * 1000:.1f} ms per learning trial")

def rate(M):
    return np.interp(np.log2(M), np.log2(list(per_trial)), list(per_trial.values()))
RUNS, TRIALS = 40, 10000
one_set = lambda M: rate(M) * TRIALS * RUNS  # s, 40 runs of 10,000 trials at one M
curve = lambda Ms: sum(one_set(M) for M in Ms)
M128, M256 = [2 ** k for k in range(8)], [2 ** k for k in range(9)]
# Lower bounds per claim: the sets of 40 runs each claim needs at the least (attempt record notes), with
# the motor control trial of 200 steps standing in for every task.
est = {
    "fig1c, fig1d: one curve, M = 1 to 128": curve(M128),
    "fig1cd, figs1d: two curves, M = 1 to 128 (before the rate search)": 2 * curve(M128),
    "fig2d, figs2a, fig2e, figs2b: one meta-learning run, 10,000 epochs of 200 trials at M = 32": rate(32) * 200 * 10000,
    "fig3ab, figs5ab, figs5ef: 3 structures x 2 tasks at M = 1 (RFLO sets only)": 6 * one_set(1),
    "fig3cd: 3 structures x 2 tasks, M = 1 to 256": 6 * curve(M256),
    "fig3ef: 3 structures x 2 tasks x 5 complexities at M = 1": 30 * one_set(1),
    "figs3c: 5 complexities x 2 rho at M = 16": 10 * one_set(16),
    "figs4cd: 5 models x 2 tasks at M = 1": 10 * one_set(1),
    "figs4ab: 8 sets at M = 1": 8 * one_set(1),
    "figs4ef: 16 sets at M = 1": 16 * one_set(1),
    "figs5gh: 12 sets at M = 1 (without the N x N update)": 12 * one_set(1),
    "fig5b: 9 sets at M = 1": 9 * one_set(1),
    "figs5j: 48 sets at M = 1": 48 * one_set(1),
    "fig4ab, fig4ef: 2 ratios x 2 tasks at N = 512 (N up to 2,048 needed)": 4 * n512 * TRIALS * RUNS,
    "fig4cd: 6 rho x 2 ratios x 2 tasks at N = 512 (N = 2,048 needed)": 24 * n512 * TRIALS * RUNS,
}
print(f"\none set of 40 runs at M = 1: {one_set(1) / 3600:.2f} h on one core")
print(pd.DataFrame({"h on one core": [v / 3600 for v in est.values()],
                    "h on four cores": [v / 3600 / 4 for v in est.values()]},
                   index=list(est)).round(1).to_string())
print("notebook limit: 1 h, the rest of this notebook included")

**Result: blocked, too large.** In this run (the cell above) one learning trial of the paper's motor
control model takes about 9 ms at M = 1 and between two and three times that at M = 128 (the fastest of three
timings), so one set of 40 runs of 10,000 trials at M = 1 takes about 1 h on one core. Every simulation claim
needs at least six such sets or their equivalent: fig3ab, the smallest, about 6 h on one core and
1.5 h on four; the curve of Figure 1 about 12 h on one core and 3 h on four; one meta-learning run of
fig2d 7 to 8 h of forward simulation before any backpropagation, with 40 runs needed. The
notebook limit is 1 h, the rest of the notebook included. The timings depend on the machine: the
verifier's copy of the previous run gave 5.3 ms at M = 1 and 18.7 ms at M = 128, the round-1 run 9
to 10 ms and 24 to 27 ms. At the fastest of these, fig3ab would take 3.5 h on one core and 0.9 h on
four, which with the 10 to 14 minutes of the rest of the notebook still exceeds 1 h; the runner's
four cores are also shared with the agent. Every other estimate in the table is larger.


## Summary

| Claim | Outcome |
|---|---|
| wm-sessions-link | partially reproduced: mouse anm00264943, 151 ALM units in 5 sessions and 72 thalamic units in 6 other sessions; 11 sessions, not 5 |
| fig6-wm-choice-r2 | not reproduced: mean test R2 at most -0.247 under the claim seed with all trials per unit (U1); medians at most 0.340, U2 means at most 0.506 (0.97 expected) |
| fig6c-wm-leading-pcs-drive-choice | partially reproduced: both readings hold under R2 (pooled), only (b) under R1 (per session); the leading PCs carry less of the readout than chance |
| figs6d-wm-thalamic-r2 | not reproduced: mean test R2 -5.16 to -1.91 (0.70 expected) |
| fig6d-wm-top5-pcs | not reproduced: no reading within 0.59 to 0.67 |
| figs6f-ct-alignment-not-significant | not reproduced: PC 1 alignment larger than readout alignment, one-sided p = 8.3e-06 |
| Six motor-task claims | blocked: data missing |
| 23 simulation claims | blocked: too large |
| figs5cd-thalamic-recurrence | blocked: methods underspecified |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The smoothing grid, the 10 ms bin and the padding are
standard choices the paper does not give. Sigma is chosen per thalamic unit on the validation R2 of
each regression: the 16-PC regression for the PC regressions, the regression on all units for that
one. The normalizations are applied to each unit's mean curve over splits and
subsamples. The noise ceilings, the larger test sets and the chance levels of the alignments are
diagnostics added on the verifier's request; none decides an outcome. P3 does not apply, as no
session holds both ALM and thalamic units. The thalamic units are regressed on ALM activity from
other sessions, as the paper's design implies; their test sets are their own sessions' trials.

What was found about the dataset. The paper's mouse is anm00264943, and its unit totals match the
paper's exactly; the thalamic units come from six sessions, not the five the paper states. The
files give no interval over which each unit was recorded, though many units fire in only part of
their session. The electrode group location gives the same coordinates (AP 2.50, ML -1.50, DV
0.00) in all 137 files, thalamic and SNr recordings included, and the thalamic recordings name no
nucleus. `photo_stim_type` is "stimulation" on every trial, including those without
photostimulation; `stim_present` and `photo_stim_period` carry the distinction. No spike train or
trial table is deposited twice, and choice decoding around delay onset is consistent with spikes
and trials sharing a clock in all eleven sessions.

In [ ]:
print(f"total time {(time.time() - started) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")